# 05 · Export the calibration artifact

Combines the thresholds fitted in notebooks 01–03 into `models/calibration.json` and records its SHA-256 digest in `models/model_manifest.json`. The browser verifies the digest before using the file (see `js/core/calibration.js`) and falls back to built-in defaults otherwise.

In [1]:
import sys, json, random, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
%matplotlib inline
sys.path.insert(0, '.')
from sway_research import *
RESULTS = DATA / 'results'; RESULTS.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'figure.dpi': 90, 'font.family': 'monospace', 'axes.facecolor': '#EDE3CC', 'figure.facecolor': '#F4ECD9'})
pd.set_option('display.width', 140)

In [2]:
import hashlib, datetime
text = json.loads((RESULTS / 'text.json').read_text()); style = json.loads((RESULTS / 'style.json').read_text())
vis = json.loads((RESULTS / 'visual.json').read_text()); ocr = json.loads((RESULTS / 'ocr.json').read_text())
cv = json.loads((RESULTS / 'vision.json').read_text())
cal = {
    'generated': datetime.date.today().isoformat(),
    'version': '1.1.0',
    'notes': 'Thresholds fitted on Project Gutenberg passages (text, style) and synthetic images (handwriting, signatures, tampering). Not validated on real casework.',
    'fusion': {'text': text['text_fusion'], 'style': style['style_fusion'], 'handwriting': vis['handwriting_fusion'], 'signature': vis['signature_fusion']},
    'handwritingScales': vis['handwriting_scales'],
    'evaluation': {'text': text['metrics'], 'style': style['metrics'], 'visual': vis['metrics'], 'ocr': ocr['metrics'], 'vision': cv},
}
models = ROOT / 'models'
blob = json.dumps(cal, indent=2, default=float).encode()
(models / 'calibration.json').write_bytes(blob)
digest = hashlib.sha256(blob).hexdigest()
manifest = {'description': 'Trusted model registry. Only artifacts listed here, with matching SHA-256 digests, are loaded by the app.',
            'artifacts': [{'file': 'calibration.json', 'kind': 'thresholds', 'sha256': digest, 'produced_by': 'notebooks/05_export_calibration.ipynb'}],
            'neural_models': []}
(models / 'model_manifest.json').write_text(json.dumps(manifest, indent=2))
print(digest); cal['fusion']

40c5227b22c4c7ce35c9ee9fd23269d545cc3b9a836b684dfe68cab60e87e350


{'text': {'modCoverage': 0.03,
  'modContainment': 0.2,
  'modParaphrase': 0.4,
  'modTfidf': 1.0,
  'highCoverage': 0.2,
  'highContainment': 0.6,
  'highSentenceRate': 0.3},
 'style': {'highAgreement': 0.5,
  'highMaxDisagreement': 0.1,
  'modAgreement': 0.4,
  'modMaxDisagreement': 0.15},
 'handwriting': {'highMeanZ': 1.601, 'modMeanZ': 2.127},
 'signature': {'highOverlap': 0.282,
  'highChamfer': 5.83,
  'modOverlap': 0.219,
  'modChamfer': 8.92}}